# NB48 — H3: Saliency-Masked Unlearning (attack the WEIGHTS, not the filter)

## Why this notebook exists
Post-hoc filtering ceiling is real and diminishing: 216 -> 211 -> 205.9.
Every prior approach kept the **poisoned teacher** generating candidates and tried to
classify poison after the fact (p ~0.65). Reaching ~139 needs p ~0.78 — a jump, not a knob.

**H3 attacks the root**: make the model itself fire fewer poison boxes, so the pool is
cleaner at the source. Bigger lever, but higher risk (retention collapse -> LB can blow past floor 260).

## Design (insured bet)
- **Saliency mask**: only the top-k% weights (by |grad| of the forget/poison loss) become
  trainable — surgical, reaches deeper than head-only WITHOUT touching retain-critical weights.
- **Forget + KD-retain + L2-SP** loss, same structure as v4 unlearner.
- **HARD retention gate** `RET_GATE_STRICT` (default 0.75): if scrubbed model's synth retention
  falls below it, the model is flagged **UNSAFE** and scrubbed exports are skipped.
- **No auto-submit.** Two export paths, both K-calibrated:
  - `H3A_source`  = M' raw detections (source suppression — the big lever)
  - `H3B_rank`    = teacher candidates ranked by M' 1-r (safer, incremental)
- **Fallback stays LB 205.94** (nb45_COMBINED_v2_Kcal). Never spend the last slot on H3.

## Read before submitting
`nb48_gate.json` -> if H3_SAFE=False, DO NOT submit any nb48_H3* file; keep 205.94.


In [ ]:
# =====================  CONFIG  =====================
CFG = dict(
    DO_UNLEARN = True,       # unlearner student (ratio r)
    DO_AMP = True,           # poison amplifier (ratio a) - the contrastive probe
    HEAD_ONLY = True, ITERS = 300, LR = 4e-5, BS_FORGET = 2, BS_RETAIN = 2,
    W_FORGET = 1.0, W_KD = 6.0, W_KD_BOX = 0.5, W_SP = 1e-4,
    FG_KD_THRESH = 0.05, FG_KD_W = 2.0, BG_KD_W = 0.02,
    AMP_ITERS = 180, AMP_LR = 6e-5,
    EVAL_EVERY = 50, GRAD_CLIP = 1.0, RET_GATE = 0.80,
    N_HELDOUT = 24, N_AUDIT_SYN = 32, POISON_PER_SCENE = 3,
    SYN_SNR = (4.0, 30.0), SYN_LEN = (24, 90), SYN_SIGMA = (1.2, 3.2),
    CAND_THRESH = 0.02, AMP_INFER = True, USE_PROTO = True,
    TAU_P = 0.85, FINAL_NMS_IOU = 0.35, TOPK_PER_IMAGE = 25,
    SEED = 42,
)
for k, v in CFG.items():
    print(f"  {k:>16} = {v}")

In [ ]:
# Kaggle's current image ships torch 2.10+cu128, whose wheels dropped Pascal (sm_60) kernels.
# P100 = sm_60 -> "no kernel image is available". T4 = sm_75 -> fine as-is.
# This cell detects the GPU BEFORE importing torch and, on Pascal, downgrades to
# torch 2.6.0+cu124 (last widely-used Kaggle pairing that ships sm_60 kernels),
# then builds detectron2 against whichever torch is active.
import subprocess

def sh(cmd, note=""):
    if note:
        print(note)
    print(">>", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = (r.stdout + r.stderr).strip().splitlines()
    print("   " + "; ".join(tail[-4:]) if tail else "   (no output)")
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-3000:])
        raise RuntimeError(f"command failed: {cmd}")

q = subprocess.run("nvidia-smi --query-gpu=compute_cap,name --format=csv,noheader",
                   shell=True, capture_output=True, text=True).stdout.strip().splitlines()
cap_s, gpu_name = ("7.5", "unknown")
if q:
    parts = [p.strip() for p in q[0].split(",")]
    cap_s = parts[0]
    gpu_name = parts[1] if len(parts) > 1 else "unknown"
try:
    CAP = float(cap_s)
except ValueError:
    CAP = 6.0 if "P100" in gpu_name.upper() else 7.5
print(f"GPU: {gpu_name} | compute capability {CAP}")

sh("pip install -q 'setuptools<81'")
if CAP < 7.0:
    sh("pip install -q torch==2.6.0 torchvision==0.21.0"
       " --index-url https://download.pytorch.org/whl/cu124",
       note="Pascal GPU: downgrading torch to a build with sm_60 kernels (~2.5 GB, 3-6 min)")
sh("pip install -q --no-cache-dir 'git+https://github.com/facebookresearch/detectron2.git'",
   note="building detectron2 against the active torch (~5-8 min)")

import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| arch list:", torch.cuda.get_arch_list())
try:
    val = float((torch.ones(8, device="cuda") * 2).sum().item())  # real kernel launch
    print(f"CUDA smoke test OK ({val}) on", torch.cuda.get_device_name(0))
except Exception as e:
    raise SystemExit(f"CUDA smoke test FAILED on {torch.cuda.get_device_name(0)}: {e}."
                     " Switch Settings -> Accelerator -> GPU T4 x2 and rerun.")

In [ ]:
import copy, glob, json, math, os, random, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import transforms as T
from detectron2.engine import DefaultPredictor
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.structures import Boxes, Instances
from detectron2.utils.events import EventStorage

import logging
for _n in ("detectron2", "fvcore", "d2"):
    logging.getLogger(_n).setLevel(logging.ERROR)

SEED = CFG["SEED"]
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_W = IMG_H = 1024

# ---- robust competition-path resolution ----
CAND_ROOTS = [
    "/kaggle/input/neural-debris-removal-in-streak-detection-models",
    "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models",
]
ROOT = next((r for r in CAND_ROOTS if os.path.isdir(r)), None)
if ROOT is None:
    hits = glob.glob("/kaggle/input/**/annotations_coco.json", recursive=True)
    assert hits, "competition data not found under /kaggle/input"
    ROOT = str(Path(hits[0]).parents[1])

pth_hits = sorted(glob.glob(str(Path(ROOT) / "**" / "*.pth"), recursive=True))
assert pth_hits, "no .pth under ROOT"
POISONED_WEIGHTS = next((p for p in pth_hits if "poison" in p.lower()), pth_hits[0])
UNLEARN_JSON = sorted(glob.glob(str(Path(ROOT) / "**" / "annotations_coco.json"), recursive=True))[0]
UNLEARN_DIR = str(Path(UNLEARN_JSON).parent)
png_dirs = {}
for p in glob.glob(str(Path(ROOT) / "**" / "*.png"), recursive=True):
    d = str(Path(p).parent); png_dirs[d] = png_dirs.get(d, 0) + 1
TEST_DIR = max(png_dirs, key=png_dirs.get)
SAMPLE_SUB = sorted(glob.glob(str(Path(ROOT) / "**" / "sample_submission.csv"), recursive=True))[0]
WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)

print("ROOT   :", ROOT)
print("weights:", POISONED_WEIGHTS)
print("unlearn:", UNLEARN_DIR, "->", len(glob.glob(UNLEARN_DIR + "/*.png")), "png")
print("test   :", TEST_DIR, "->", png_dirs[TEST_DIR], "png")
print("sample :", SAMPLE_SUB)

def iou_matrix(a, b):
    # a: (N,4) XYXY, b: (M,4) XYXY -> (N,M)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    a = np.asarray(a, np.float32); b = np.asarray(b, np.float32)
    ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    iw = np.maximum(0, np.minimum(ax2, bx2) - np.maximum(ax1, bx1))
    ih = np.maximum(0, np.minimum(ay2, by2) - np.maximum(ay1, by1))
    inter = iw * ih
    ua = np.maximum(0, ax2 - ax1) * np.maximum(0, ay2 - ay1)
    ub = np.maximum(0, bx2 - bx1) * np.maximum(0, by2 - by1)
    union = ua + ub - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)

def to3(g):
    return np.repeat(g[:, :, None], 3, axis=2)

In [ ]:
# Architecture + preprocessing identical to the official baseline (do not change).
BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1

def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    assert im is not None, path
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    else:
        im = im.astype(np.float32) / 255.0
    im = np.clip(im * 255.0, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im  # HWC float32 in [0,255]

def load_image1(path):
    return load_image(path)[:, :, 0].copy()

def build_cfg(weights, score_thresh):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = str(weights)
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    return cfg

class FeatPredictor:
    # DefaultPredictor + optional P3 FPN feature capture (for prototype scoring)
    def __init__(self, weights, score_thresh, capture=False):
        self.pred = DefaultPredictor(build_cfg(weights, score_thresh))
        self.capture = capture
        self._p3 = None
        if capture:
            self.hook = self.pred.model.backbone.register_forward_hook(self._grab)
    def _grab(self, module, inputs, output):
        self._p3 = output["p3"].detach()
    def __call__(self, img):
        with torch.autocast("cuda", enabled=(CFG["AMP_INFER"] and DEVICE == "cuda")):
            out = self.pred(img)
        inst = out["instances"].to("cpu")
        return (inst.pred_boxes.tensor.numpy().astype(np.float32),
                inst.scores.numpy().astype(np.float32))
    def pool_feats(self, boxes_xyxy):
        # mean-pool the captured P3 map (stride 8, resized coords) inside each original-coord box
        f = self._p3[0].float()
        C, Hf, Wf = f.shape
        s = (Hf * 8.0) / IMG_H  # resized/original scale
        out = np.zeros((len(boxes_xyxy), C), np.float32)
        for i, (x1, y1, x2, y2) in enumerate(boxes_xyxy):
            fx1 = int(np.clip(np.floor(x1 * s / 8.0), 0, Wf - 1))
            fy1 = int(np.clip(np.floor(y1 * s / 8.0), 0, Hf - 1))
            fx2 = int(np.clip(np.ceil(x2 * s / 8.0), fx1 + 1, Wf))
            fy2 = int(np.clip(np.ceil(y2 * s / 8.0), fy1 + 1, Hf))
            v = f[:, fy1:fy2, fx1:fx2].mean(dim=(1, 2)).cpu().numpy()
            out[i] = v / (np.linalg.norm(v) + 1e-6)
        return out

In [ ]:
with open(UNLEARN_JSON) as f:
    coco = json.load(f)
id2im = {im["id"]: im for im in coco["images"]}
poison_anns = {}
for ann in coco["annotations"]:
    fn = id2im[ann["image_id"]]["file_name"]
    poison_anns.setdefault(fn, []).append(list(ann["bbox"]))  # XYWH
unlearn_files = sorted(Path(UNLEARN_DIR).glob("*.png"))
n_boxes = sum(len(v) for v in poison_anns.values())
whs = np.array([b[2:4] for v in poison_anns.values() for b in v], np.float32)
print(f"{len(unlearn_files)} unlearn images | {n_boxes} poison boxes")
print("w percentiles :", np.percentile(whs[:, 0], [0, 25, 50, 75, 100]).round(1))
print("h percentiles :", np.percentile(whs[:, 1], [0, 25, 50, 75, 100]).round(1))
print("aspect w/h    :", np.percentile(whs[:, 0] / np.maximum(whs[:, 1], 1), [0, 50, 100]).round(2))

# Mahalanobis geometry over log(w,h) of poison boxes (weak prior, small blend weight)
logwh = np.log(np.maximum(whs, 1e-3))
GEO_MU = logwh.mean(0)
GEO_COV_INV = np.linalg.inv(np.cov(logwh.T) + 1e-3 * np.eye(2))
def geo_score(boxes_xyxy):
    if len(boxes_xyxy) == 0:
        return np.zeros(0, np.float32)
    wh = np.stack([boxes_xyxy[:, 2] - boxes_xyxy[:, 0], boxes_xyxy[:, 3] - boxes_xyxy[:, 1]], 1)
    d = np.log(np.maximum(wh, 1e-3)) - GEO_MU
    m2 = np.einsum("ni,ij,nj->n", d, GEO_COV_INV, d)
    return np.exp(-0.25 * m2).astype(np.float32)

# LOOK AT THIS GRID: this is the trigger being unlearned. If these crops are visually
# indistinguishable from normal streaks, expect weaker generalization from every component.
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
k = 0
for fp in unlearn_files:
    g = None
    for (x, y, w, h) in poison_anns.get(fp.name, []):
        if k >= 12:
            break
        if g is None:
            g = load_image1(fp)
        pad = 24
        x1, y1 = int(max(0, x - pad)), int(max(0, y - pad))
        x2, y2 = int(min(IMG_W, x + w + pad)), int(min(IMG_H, y + h + pad))
        ax = axes[k // 4][k % 4]
        ax.imshow(g[y1:y2, x1:x2], cmap="gray")
        ax.set_title(f"{fp.name[:16]}  {int(w)}x{int(h)}", fontsize=8)
        ax.axis("off")
        k += 1
    if k >= 12:
        break
for j in range(k, 12):
    axes[j // 4][j % 4].axis("off")
plt.suptitle("poison GT crops (the trigger)"); plt.tight_layout(); plt.show()

In [ ]:
# The teacher (poisoned model) MUST detect the poison GT with high confidence.
# If recall here is low, the preprocessing does not match training: stop and fix first.
teacher = FeatPredictor(POISONED_WEIGHTS, CFG["CAND_THRESH"], capture=CFG["USE_PROTO"])

hits, confs = 0, []
for fp in tqdm(unlearn_files, desc="teacher @ unlearn"):
    boxes, scores = teacher(load_image(fp))
    gts = np.array([[x, y, x + w, y + h] for (x, y, w, h) in poison_anns.get(fp.name, [])], np.float32)
    if len(gts) == 0:
        continue
    m = iou_matrix(gts, boxes)
    for i in range(len(gts)):
        if m.shape[1] and m[i].max() >= 0.5:
            hits += 1
            confs.append(float(scores[m[i].argmax()]))
print(f"poisoned-model recall@0.5 on poison GT: {hits}/{n_boxes}"
      f" | mean conf {np.mean(confs) if confs else 0:.3f}")
assert hits >= 0.7 * n_boxes, "loader/config mismatch vs training - do not proceed"

In [ ]:
# 1-channel float32 scenes generated on the fly (RAM-safe). to3() at model boundary.
rng = np.random.default_rng(SEED)

def inpaint1(g, boxes_xywh, pad=6):
    g = g.copy()
    for (x, y, w, h) in boxes_xywh:
        x1, y1 = int(max(0, x - pad)), int(max(0, y - pad))
        x2, y2 = int(min(IMG_W, x + w + pad)), int(min(IMG_H, y + h + pad))
        rx1, ry1 = max(0, x1 - 32), max(0, y1 - 32)
        rx2, ry2 = min(IMG_W, x2 + 32), min(IMG_H, y2 + 32)
        ring = g[ry1:ry2, rx1:rx2]
        med = float(np.median(ring))
        sig = 1.4826 * float(np.median(np.abs(ring - med))) + 1e-3
        g[y1:y2, x1:x2] = np.clip(
            np.random.normal(med, sig, (y2 - y1, x2 - x1)), 0, 255).astype(np.float32)
    return g

BG_BANK = [inpaint1(load_image1(fp), poison_anns.get(fp.name, [])) for fp in unlearn_files]
ORIGINALS = [
    (load_image1(fp),
     np.array([[x, y, x + w, y + h] for (x, y, w, h) in poison_anns.get(fp.name, [])], np.float32))
    for fp in unlearn_files
]

PATCHES = []
for fp in unlearn_files:
    g = load_image1(fp)
    for (x, y, w, h) in poison_anns.get(fp.name, []):
        pad = 10
        x1, y1 = int(max(0, x - pad)), int(max(0, y - pad))
        x2, y2 = int(min(IMG_W, x + w + pad)), int(min(IMG_H, y + h + pad))
        PATCHES.append(g[y1:y2, x1:x2].copy())
print("poison patch bank:", len(PATCHES), "| background bank:", len(BG_BANK))

def rand_bg1(rng):
    b = BG_BANK[int(rng.integers(0, len(BG_BANK)))]
    b = np.rot90(b, int(rng.integers(0, 4)))
    if rng.random() < 0.5:
        b = b[:, ::-1]
    return np.ascontiguousarray(b)

def paste_poison1(bg, rng):
    # additive-light blend: signal = patch - its own background level (no rectangle seams).
    # rot90/flip + mild scale only: arbitrary-angle warps could smear a pixel-precise trigger.
    g = bg.copy()
    boxes = []
    for _ in range(int(rng.integers(1, CFG["POISON_PER_SCENE"] + 1))):
        p = PATCHES[int(rng.integers(0, len(PATCHES)))].astype(np.float32)
        p = np.rot90(p, int(rng.integers(0, 4)))
        if rng.random() < 0.5:
            p = p[:, ::-1]
        sc = float(rng.uniform(0.85, 1.2))
        p = cv2.resize(p, (max(8, int(p.shape[1] * sc)), max(8, int(p.shape[0] * sc))))
        p = p * float(rng.uniform(0.8, 1.25))
        ph, pw = p.shape
        if ph >= IMG_H - 2 or pw >= IMG_W - 2:
            continue
        y0 = int(rng.integers(1, IMG_H - ph - 1))
        x0 = int(rng.integers(1, IMG_W - pw - 1))
        sig = np.clip(p - float(np.median(p)), 0, None)
        g[y0:y0 + ph, x0:x0 + pw] = np.clip(g[y0:y0 + ph, x0:x0 + pw] + sig, 0, 255)
        boxes.append([x0, y0, x0 + pw, y0 + ph])
    return g, (np.array(boxes, np.float32) if boxes else np.zeros((0, 4), np.float32))

def synth1(bg, rng):
    # synthetic clean streaks: AA line + gaussian PSF; SYN_LEN now spans SHORT (trigger-size) to long
    g = bg.copy()
    med = float(np.median(g))
    nsig = 1.4826 * float(np.median(np.abs(g - med))) + 1e-3
    boxes = []
    for _ in range(int(rng.integers(1, 3))):
        L = float(rng.uniform(*CFG["SYN_LEN"]))
        ang = float(rng.uniform(0, np.pi))
        cx, cy = float(rng.uniform(80, IMG_W - 80)), float(rng.uniform(80, IMG_H - 80))
        dx, dy = 0.5 * L * np.cos(ang), 0.5 * L * np.sin(ang)
        x1, y1, x2, y2 = cx - dx, cy - dy, cx + dx, cy + dy
        sgm = float(rng.uniform(*CFG["SYN_SIGMA"]))
        snr = float(rng.uniform(*CFG["SYN_SNR"]))
        canvas = np.zeros((IMG_H, IMG_W), np.float32)
        cv2.line(canvas, (int(x1), int(y1)), (int(x2), int(y2)), 1.0, 1, cv2.LINE_AA)
        canvas = cv2.GaussianBlur(canvas, (0, 0), sgm)
        mx = canvas.max()
        if mx > 0:
            canvas /= mx
        g = np.clip(g + canvas * snr * nsig, 0, 255)
        bx1, by1 = max(0.0, min(x1, x2) - 3 * sgm), max(0.0, min(y1, y2) - 3 * sgm)
        bx2, by2 = min(float(IMG_W), max(x1, x2) + 3 * sgm), min(float(IMG_H), max(y1, y2) + 3 * sgm)
        boxes.append([bx1, by1, bx2, by2])
    return g, np.array(boxes, np.float32)

def sample_forget(rng):
    # V8 collapse fix: 60% MIXED discriminative scenes - pasted triggers stay UNLABELED (background)
    # while synthetic clean streaks are the GT positives. The model must separate, not silence.
    r = rng.random()
    if r < 0.15:
        g, _ = ORIGINALS[int(rng.integers(0, len(ORIGINALS)))]
        return to3(g), np.zeros((0, 4), np.float32)
    if r < 0.40:
        g, _ = paste_poison1(rand_bg1(rng), rng)
        return to3(g), np.zeros((0, 4), np.float32)
    g, _pb = paste_poison1(rand_bg1(rng), rng)
    g, sb = synth1(g, rng)
    return to3(g), sb

def sample_retain(rng):
    # trigger-free scenes for KD: 2/3 synthetic streaks, 1/3 plain backgrounds
    if rng.random() < (1.0 / 3.0):
        return to3(rand_bg1(rng))
    g, _ = synth1(rand_bg1(rng), rng)
    return to3(g)

HELDOUT = [paste_poison1(rand_bg1(rng), rng) for _ in range(CFG["N_HELDOUT"])]
AUDIT_SYN = [synth1(rand_bg1(rng), rng) for _ in range(CFG["N_AUDIT_SYN"])]

mg, mpb = paste_poison1(rand_bg1(rng), rng)
mg, msb = synth1(mg, rng)
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
panels = [
    (HELDOUT[0][0], [("r", HELDOUT[0][1])], "heldout: pure pasted triggers"),
    (mg, [("r", mpb), ("lime", msb)], "mixed forget: triggers(red)=bg, synth(green)=GT"),
    (AUDIT_SYN[0][0], [("lime", AUDIT_SYN[0][1])], "retain/audit: synthetic streaks"),
    (BG_BANK[0], [], "retain: inpainted background"),
]
for ax, (im, boxsets, title) in zip(axes, panels):
    ax.imshow(im, cmap="gray")
    for color, bxs in boxsets:
        for (x1, y1, x2, y2) in bxs:
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, lw=1))
    ax.set_title(title, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Teacher hit-rate on synthetic streaks, split by length. Short streaks (<90 px) live in the
# trigger-size regime: they are BOTH the proto negatives that make demotion trigger-specific
# (not just size-specific) AND the retention canaries for training.
det_flags, det_confs, det_lens = [], [], []
teacher_syn = []
for g, bxs in tqdm(AUDIT_SYN, desc="teacher @ synth"):
    boxes, scores = teacher(to3(g))
    keep = scores >= 0.2
    teacher_syn.append((boxes[keep], scores[keep]))
    m = iou_matrix(bxs, boxes)
    for i in range(len(bxs)):
        ok = bool(m.shape[1]) and m[i].max() >= 0.3
        det_flags.append(float(ok))
        det_lens.append(float(max(bxs[i, 2] - bxs[i, 0], bxs[i, 3] - bxs[i, 1])))
        if ok:
            det_confs.append(float(scores[m[i].argmax()]))
det_flags = np.array(det_flags); det_lens = np.array(det_lens)
SYN_HIT = float(det_flags.mean()) if len(det_flags) else 0.0
short = det_lens < 90
print(f"teacher hit-rate on synthetic streaks: {SYN_HIT:.2f}"
      f" | mean conf {np.mean(det_confs) if det_confs else 0:.2f}")
if short.any():
    print(f"  short (<90px): {det_flags[short].mean():.2f} over {int(short.sum())}"
          f" | long: {det_flags[~short].mean():.2f} over {int((~short).sum())}")
if SYN_HIT < 0.5:
    print("WARNING: synthetic streaks off-distribution; proto negatives and retention audit degraded")

In [ ]:
# Poison prototype scoring in the teacher feature space: LDA over pooled P3 features,
# positives = poison GT + pasted views, negatives = teacher detections on synthetic streaks.
PROTO = None
if CFG["USE_PROTO"]:
    pos_feats = []
    for fp in unlearn_files:
        _ = teacher(load_image(fp))
        gts = np.array([[x, y, x + w, y + h] for (x, y, w, h) in poison_anns.get(fp.name, [])], np.float32)
        if len(gts):
            pos_feats.append(teacher.pool_feats(gts))
    for _ in range(60):
        g, bxs = paste_poison1(rand_bg1(rng), rng)
        if len(bxs) == 0:
            continue
        _ = teacher(to3(g))
        pos_feats.append(teacher.pool_feats(bxs))
    neg_feats = []
    for (g, _), (tb, ts) in zip(AUDIT_SYN, teacher_syn):
        good = ts >= 0.3
        if good.any():
            _ = teacher(to3(g))
            neg_feats.append(teacher.pool_feats(tb[good]))
    if SYN_HIT < 0.4 or not neg_feats:
        print("proto disabled: no reliable clean prototypes")
        CFG["USE_PROTO"] = False
    else:
        pos = np.concatenate(pos_feats, 0); neg = np.concatenate(neg_feats, 0)
        mu_p, mu_n = pos.mean(0), neg.mean(0)
        Xc = np.concatenate([pos - mu_p, neg - mu_n], 0)
        S = (Xc.T @ Xc) / max(len(Xc) - 1, 1)
        S = S + (0.5 * np.trace(S) / S.shape[0] + 1e-4) * np.eye(S.shape[0], dtype=S.dtype)
        wvec = np.linalg.solve(S, (mu_p - mu_n).astype(np.float64)).astype(np.float32)
        zb = -0.5 * float((mu_p + mu_n) @ wvec)
        zp, zn = pos @ wvec + zb, neg @ wvec + zb
        zs = 0.5 * (zp.std() + zn.std()) + 1e-6
        print(f"proto fit: {len(pos)} pos / {len(neg)} neg"
              f" | z_pos {zp.mean()/zs:+.2f} z_neg {zn.mean()/zs:+.2f} (want well separated)")
        def PROTO(feats):
            z = (feats @ wvec + zb) / zs
            return (1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))).astype(np.float32)
        if abs(zp.mean() - zn.mean()) / zs < 1.0:
            print("WARNING: weak proto separation -> consider W_PROTO=0 on the next A/B run")

## Shared helpers (to_batched / head_out / infer_np / audits) — from v4


In [ ]:
resize_aug = T.ResizeShortestEdge(short_edge_length=800, max_size=1333)  # mirrors DefaultPredictor

def to_batched(img3, boxes_xyxy=None, train=True):
    tfm = resize_aug.get_transform(img3)
    img_r = tfm.apply_image(img3)
    d = {"image": torch.as_tensor(np.ascontiguousarray(img_r.transpose(2, 0, 1))),
         "height": IMG_H, "width": IMG_W}
    if train:
        h, w = img_r.shape[:2]
        inst = Instances((h, w))
        if boxes_xyxy is None or len(boxes_xyxy) == 0:
            inst.gt_boxes = Boxes(torch.zeros((0, 4), dtype=torch.float32))
            inst.gt_classes = torch.zeros((0,), dtype=torch.int64)
        else:
            b = tfm.apply_box(np.asarray(boxes_xyxy, np.float32))
            inst.gt_boxes = Boxes(torch.as_tensor(b, dtype=torch.float32))
            inst.gt_classes = torch.zeros((len(b),), dtype=torch.int64)
        d["instances"] = inst
    return d

def head_out(model, batched):
    images = model.preprocess_image(batched)
    feats = model.backbone(images.tensor)
    feats = [feats[f] for f in model.head_in_features]
    return model.head(feats)

@torch.no_grad()
def infer_np(model, img3):
    was_training = model.training
    model.eval()
    with torch.autocast("cuda", enabled=(CFG["AMP_INFER"] and DEVICE == "cuda")):
        out = model([to_batched(img3, train=False)])[0]["instances"].to("cpu")
    if was_training:
        model.train()
    return (out.pred_boxes.tensor.numpy().astype(np.float32),
            out.scores.numpy().astype(np.float32))

@torch.no_grad()
def audit_suppression(model, scenes):
    vals = []
    for g, bxs in scenes:
        boxes, scores = infer_np(model, to3(g))
        m = iou_matrix(bxs, boxes)
        for i in range(len(bxs)):
            sel = (m[i] >= 0.3) if m.shape[1] else np.zeros(0, bool)
            vals.append(float(scores[sel].max()) if sel.any() else 0.0)
    return float(np.mean(vals)) if vals else 0.0

@torch.no_grad()
def audit_retention(model, scenes, refs):
    vals = []
    for (g, _), (tb, ts) in zip(scenes, refs):
        if len(tb) == 0:
            continue
        boxes, scores = infer_np(model, to3(g))
        m = iou_matrix(tb, boxes)
        for i in range(len(tb)):
            if m.shape[1] and m[i].max() >= 0.5:
                vals.append(min(1.5, float(scores[m[i].argmax()]) / max(float(ts[i]), 1e-6)))
            else:
                vals.append(0.0)
    return float(np.mean(vals)) if vals else 1.0


## H3 config


In [ ]:
# ===================== H3 CONFIG =====================
CFG["DO_AMP"] = False   # not needed here

H3 = dict(
    MASK_TOPK   = 0.10,     # fraction of in-scope weights (by |forget-grad|) to unfreeze
    SCOPE       = "backbone_head",  # head + backbone.fpn + backbone.bottom_up top blocks (NOT stem)
    ITERS       = 400,
    LR          = 2e-5,
    BS_FORGET   = 2,
    BS_RETAIN   = 2,
    W_FORGET    = 1.0,
    W_KD        = 8.0,      # heavier KD than v4 (protect retention when scrubbing deeper)
    W_KD_BOX    = 0.5,
    W_SP        = 3e-4,     # stronger L2-SP anchor (deeper edits need more regularization)
    FG_KD_THRESH= 0.05,
    FG_KD_W     = 2.5,
    BG_KD_W     = 0.02,
    GRAD_CLIP   = 1.0,
    EVAL_EVERY  = 50,
    SAL_BATCHES = 24,       # batches to accumulate saliency
    RET_GATE_STRICT = 0.75, # HARD gate: scrubbed model must retain >= this on synth
    SUPP_TARGET     = 0.35, # want residual poison conf below this (soft target for ckpt select)
)
for k,v in H3.items():
    print(f"  {k:>16} = {v}")


## H3 saliency mask + masked unlearn


In [ ]:
# ===================== H3: compute saliency mask, then masked unlearn =====================
def in_scope(n):
    if n.startswith("head."):
        return True
    if n.startswith("backbone.fpn"):
        return True
    # top of bottom_up (res5) only; keep stem/res2-4 frozen for stability
    if n.startswith("backbone.bottom_up.res5"):
        return True
    return False

cfg_h = build_cfg(POISONED_WEIGHTS, CFG["CAND_THRESH"])
scrub = build_model(cfg_h)
DetectionCheckpointer(scrub).load(str(POISONED_WEIGHTS))
scrub.to(DEVICE)
teacher_m = teacher.pred.model

scope_params = [(n, p) for n, p in scrub.named_parameters() if in_scope(n)]
print(f"in-scope params: {sum(p.numel() for _,p in scope_params)/1e6:.1f}M "
      f"({len(scope_params)} tensors)")

# ---- 1) accumulate |grad| of the FORGET (poison) loss to find poison-salient weights ----
scrub.train()
accum = {n: torch.zeros_like(p, device=DEVICE) for n, p in scope_params}
for _ in range(H3["SAL_BATCHES"]):
    fb = [sample_forget(rng) for _ in range(H3["BS_FORGET"])]
    batch_f = [to_batched(img, gtb, train=True) for img, gtb in fb]
    with EventStorage(0):
        ld = scrub(batch_f)
    loss_f = ld["loss_cls"] + ld.get("loss_box_reg", 0)
    grads = torch.autograd.grad(loss_f, [p for _, p in scope_params],
                                retain_graph=False, allow_unused=True)
    for (n, _), g in zip(scope_params, grads):
        if g is not None:
            accum[n] += g.abs()
    scrub.zero_grad(set_to_none=True)

# global top-k threshold over all in-scope grads
all_vals = torch.cat([v.flatten() for v in accum.values()])
thr = torch.quantile(all_vals.float(), 1.0 - H3["MASK_TOPK"])
mask = {n: (accum[n] >= thr).float() for n, _ in scope_params}
n_on = int(sum(m.sum().item() for m in mask.values()))
n_tot = int(sum(m.numel() for m in mask.values()))
print(f"saliency mask: {n_on}/{n_tot} weights active ({100*n_on/max(n_tot,1):.1f}%) thr={float(thr):.3e}")

# ---- 2) masked unlearn: only salient weights update ----
for n, p in scrub.named_parameters():
    p.requires_grad_(in_scope(n))
trainable = [p for n, p in scrub.named_parameters() if in_scope(n)]
anchor0 = {n: p.detach().clone() for n, p in scrub.named_parameters() if in_scope(n)}

opt = torch.optim.AdamW(trainable, lr=H3["LR"], weight_decay=0.0)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=H3["ITERS"])
best = {"score": -9.0, "it": -1, "state": None, "supp": 1.0, "ret": 0.0}

t0 = time.time()
for it in range(1, H3["ITERS"] + 1):
    scrub.train()
    fb = [sample_forget(rng) for _ in range(H3["BS_FORGET"])]
    batch_f = [to_batched(img, gtb, train=True) for img, gtb in fb]
    with EventStorage(it):
        ld = scrub(batch_f)
    loss_forget = ld["loss_cls"] + ld.get("loss_box_reg", 0)

    batch_r = [to_batched(sample_retain(rng), train=False) for _ in range(H3["BS_RETAIN"])]
    with torch.no_grad():
        tl, td = head_out(teacher_m, batch_r)
    sl, sd = head_out(scrub, batch_r)
    kd_cls = torch.zeros((), device=DEVICE); kd_box = torch.zeros((), device=DEVICE)
    wsum = torch.zeros((), device=DEVICE)
    for s_l, t_l, s_d, t_d in zip(sl, tl, sd, td):
        tp = torch.sigmoid(t_l)
        w = torch.full_like(tp, H3["BG_KD_W"]); w[tp > H3["FG_KD_THRESH"]] = H3["FG_KD_W"]
        bce = F.binary_cross_entropy_with_logits(s_l, tp, reduction="none")
        kd_cls = kd_cls + (w * bce).sum(); wsum = wsum + w.sum()
        w4 = tp.detach().repeat_interleave(4, dim=1)
        kd_box = kd_box + (w4 * (s_d - t_d).abs()).sum() / (w4.sum() + 1e-6)
    kd_cls = kd_cls / (wsum + 1e-6)

    sp = torch.zeros((), device=DEVICE)
    for n, p in scrub.named_parameters():
        if p.requires_grad:
            sp = sp + ((p - anchor0[n]) ** 2).sum()
    loss = (H3["W_FORGET"] * loss_forget + H3["W_KD"] * kd_cls +
            H3["W_KD_BOX"] * kd_box + H3["W_SP"] * sp)

    opt.zero_grad(set_to_none=True)
    loss.backward()
    # ---- apply saliency mask to gradients: freeze non-salient weights ----
    for n, p in scrub.named_parameters():
        if p.requires_grad and n in mask and p.grad is not None:
            p.grad.mul_(mask[n])
    torch.nn.utils.clip_grad_norm_(trainable, H3["GRAD_CLIP"])
    opt.step(); sched.step()

    if it % H3["EVAL_EVERY"] == 0 or it == H3["ITERS"]:
        supp = audit_suppression(scrub, HELDOUT[:20])
        ret = audit_retention(scrub, AUDIT_SYN[:20], teacher_syn[:20])
        # ckpt score: reward low poison conf, but ONLY among retention-safe checkpoints
        score = (ret - supp) if ret >= H3["RET_GATE_STRICT"] else (ret - supp - 1.0)
        print(f"[H3 {it:4d}/{H3['ITERS']}] forget {float(loss_forget.detach()):.3f}"
              f" kd {float(kd_cls.detach()):.4f} | supp {supp:.3f} ret {ret:.3f}"
              f" | {time.time()-t0:.0f}s")
        if score > best["score"]:
            best = {"score": score, "it": it, "supp": supp, "ret": ret,
                    "state": {k: v.detach().cpu().clone() for k, v in scrub.state_dict().items()}}

if best["state"] is not None:
    scrub.load_state_dict(best["state"])
    print(f"restored best H3 ckpt: iter {best['it']} supp={best['supp']:.3f} ret={best['ret']:.3f}")
scrub.eval()
torch.save({"model": scrub.state_dict()}, WORK / "nb48_scrub.pth")


## H3 hard retention gate


In [ ]:
# ===================== H3 HARD RETENTION GATE =====================
import json as _json
scrub.eval()
supp_full = audit_suppression(scrub, HELDOUT)
ret_full  = audit_retention(scrub, AUDIT_SYN, teacher_syn)

# teacher baselines for reference
supp_teacher = audit_suppression(teacher.pred.model, HELDOUT)
print(f"scrub : residual poison conf {supp_full:.3f} | synth retention {ret_full:.3f}")
print(f"teacher poison conf (ref)   {supp_teacher:.3f}")
print(f"poison suppressed by        {supp_teacher - supp_full:+.3f} (want strongly negative residual)")

H3_SAFE = bool(ret_full >= H3["RET_GATE_STRICT"])
POISON_CUT = bool(supp_full <= H3["SUPP_TARGET"])
print(f"\nRET_GATE_STRICT = {H3['RET_GATE_STRICT']}  -> H3_SAFE = {H3_SAFE}")
print(f"SUPP_TARGET     = {H3['SUPP_TARGET']}  -> POISON_CUT = {POISON_CUT}")

gate = {
    "ret_full": ret_full, "supp_full": supp_full,
    "supp_teacher": supp_teacher,
    "RET_GATE_STRICT": H3["RET_GATE_STRICT"], "SUPP_TARGET": H3["SUPP_TARGET"],
    "H3_SAFE": H3_SAFE, "POISON_CUT": POISON_CUT,
    "mask_topk": H3["MASK_TOPK"], "iters": H3["ITERS"],
}
with open(WORK / "nb48_gate.json", "w") as f:
    _json.dump(gate, f, indent=2)
print("saved", WORK / "nb48_gate.json")

if not H3_SAFE:
    print("\n*** H3 UNSAFE: retention below gate. Scrubbed exports will be SKIPPED. ***")
    print("*** Keep fallback nb45_COMBINED_v2_Kcal (LB 205.94). ***")


## H3 test inference + exports


In [ ]:
# ===================== H3 test inference + K-calibrated exports (no auto-submit) =====================
def stem_key(p):
    try: return (0, int(p.stem))
    except ValueError: return (1, p.stem)

def norm_id(x):
    try: return str(int(str(x)))
    except ValueError: return str(x)

def fmt(bx, sc):
    parts = []
    for (x1, y1, x2, y2), s in zip(bx, sc):
        x1 = float(np.clip(x1,0,IMG_W)); y1=float(np.clip(y1,0,IMG_H))
        x2 = float(np.clip(x2,0,IMG_W)); y2=float(np.clip(y2,0,IMG_H))
        w,h = x2-x1, y2-y1
        if w>0 and h>0 and s>0:
            parts += [f"{s:.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"]
    return " ".join(parts) or " "

def match_ratio(b, s, db, ds):
    r = np.zeros(len(b), np.float32)
    if len(db) and len(b):
        m = iou_matrix(b, db); bi, bv = m.argmax(1), m.max(1); ok = bv >= 0.5
        r[ok] = np.clip(ds[bi[ok]] / np.maximum(s[ok], 1e-6), 0, 2.5)
    return r

BASE_THR, EPS_Q, BOOST, MAX_KEEP = 0.10, 0.02, 0.15, 2
TARGET_K, S_MIN = 450, 0.30

test_files = sorted(Path(TEST_DIR).glob("*.png"), key=stem_key)
sample = pd.read_csv(SAMPLE_SUB, dtype={"image_id": str})
print(len(test_files), "test images")

# teacher candidates + scrub ratio (H3B), and scrub raw detections (H3A)
cand_t, ratio_scrub = {}, {}
cand_scrub = {}
for fp in tqdm(test_files, desc="H3 inference"):
    img = load_image(fp)
    tb, ts = teacher(img)
    cand_t[fp.stem] = (tb, ts)
    sb, ss = infer_np(scrub, img)
    cand_scrub[fp.stem] = (sb, ss)
    ratio_scrub[fp.stem] = (match_ratio(tb, ts, sb, ss) if len(tb) else np.zeros(0, np.float32))

manifest = []

def calibrate_and_compose_rank(name, score_dict, cand_src, target_k=TARGET_K):
    # keep candidates with score < thr (score high = poison-like), K-calibrated
    all_sc = []
    for stem,(b,s) in cand_src.items():
        idx = np.where(s>=BASE_THR)[0]
        if len(idx): all_sc.append(score_dict[stem][idx][s[idx]>=S_MIN])
    all_sc = np.concatenate(all_sc) if all_sc else np.array([0.5],np.float32)
    grid = np.unique(np.quantile(all_sc, np.linspace(0.02,0.98,60)))
    best_t,best_d,bestK=None,1e18,None
    for t in grid:
        K=0
        for stem,(b,s) in cand_src.items():
            idx=np.where(s>=BASE_THR)[0]; sc=s[idx]; scv=score_dict[stem][idx]
            keep=(scv<t)&(sc>=S_MIN)
            K+=min(int(keep.sum()),MAX_KEEP)
        d=abs(K-target_k)
        if d<best_d: best_d,best_t,bestK=d,float(t),K
    # compose
    preds={}; kK=0; kmass=0.0
    for stem,(b,s) in cand_src.items():
        idx=np.where(s>=BASE_THR)[0]; kb=b[idx]; sc=s[idx]; scv=score_dict[stem][idx]
        q=np.full(len(idx),EPS_Q,np.float32)
        keep=(scv<best_t)&(sc>=S_MIN)
        if keep.sum()>MAX_KEEP:
            el=np.flatnonzero(keep); allow=el[np.argsort(scv[el])[:MAX_KEEP]]
            out=np.zeros_like(keep,bool); out[allow]=True; keep=keep&out
        q[keep]=np.minimum(1.0, sc[keep]+BOOST)
        kK+=int(keep.sum()); kmass+=float(q[keep].sum())
        preds[norm_id(stem)]=(kb,q)
    sub=sample.copy()
    sub["prediction_string"]=sub["image_id"].map(lambda i: fmt(*preds.get(norm_id(i),(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))))
    path=WORK/f"nb48_{name}.csv"; sub.to_csv(path,index=False)
    row={"name":name,"keepK":kK,"kmass":round(kmass,1),"thr":round(best_t,4),"path":str(path)}
    print(row); return row

def calibrate_and_compose_source(name, cand_src, target_k=TARGET_K):
    # M' raw detections: keep highest-confidence boxes, K-calibrate by conf threshold
    all_sc=[]
    for stem,(b,s) in cand_src.items():
        if len(s): all_sc.append(s)
    all_sc=np.concatenate(all_sc) if all_sc else np.array([0.3],np.float32)
    grid=np.unique(np.quantile(all_sc,np.linspace(0.02,0.98,80)))
    best_t,best_d,bestK=None,1e18,None
    for t in grid:
        K=0
        for stem,(b,s) in cand_src.items():
            keep=s>=t; K+=min(int(keep.sum()),MAX_KEEP)
        d=abs(K-target_k)
        if d<best_d: best_d,best_t,bestK=d,float(t),K
    preds={}; kK=0; kmass=0.0
    for stem,(b,s) in cand_src.items():
        keep=s>=best_t
        if keep.sum()>MAX_KEEP:
            el=np.flatnonzero(keep); allow=el[np.argsort(-s[el])[:MAX_KEEP]]
            out=np.zeros_like(keep,bool); out[allow]=True; keep=keep&out
        kb=b[keep]; q=np.minimum(1.0, s[keep]+BOOST)
        kK+=int(keep.sum()); kmass+=float(q.sum())
        preds[norm_id(stem)]=(kb,q)
    sub=sample.copy()
    sub["prediction_string"]=sub["image_id"].map(lambda i: fmt(*preds.get(norm_id(i),(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))))
    path=WORK/f"nb48_{name}.csv"; sub.to_csv(path,index=False)
    row={"name":name,"keepK":kK,"kmass":round(kmass,1),"thr":round(best_t,4),"path":str(path)}
    print(row); return row

if H3_SAFE:
    print("H3_SAFE -> exporting scrubbed variants")
    # H3B: teacher candidates ranked by scrub 1-r (deeper-scrubbed ranker)
    score_1mr_scrub = {stem: (1.0 - ratio_scrub[stem]).astype(np.float32) for stem in cand_t}
    manifest.append(calibrate_and_compose_rank("H3B_scrubrank_Kcal", score_1mr_scrub, cand_t))
    # H3A: source suppression -> M' raw detections
    manifest.append(calibrate_and_compose_source("H3A_source_Kcal", cand_scrub))
else:
    print("H3 UNSAFE -> NOT exporting scrubbed variants. Keep 205.94 fallback.")

adf = pd.DataFrame(manifest)
print("\n=== NB48 EXPORT MANIFEST ===")
print(adf.to_string(index=False) if len(adf) else "(none - unsafe)")
if len(adf): adf.to_csv(WORK/"nb48_manifest.csv", index=False)

print("\n=== SUBMIT POLICY ===")
print("Fallback (final safety): nb45_COMBINED_v2_Kcal.csv = LB 205.94")
print("If H3_SAFE:")
print("  - Try H3A_source first (biggest lever: cleaner pool at source).")
print("  - H3B_scrubrank = safer incremental (teacher cand, deeper ranker).")
print("  - NEVER use the last pre-deadline slot on H3. Keep >=1 spare to revert.")
print("If NOT H3_SAFE: do not submit any nb48 file.")
